In [1]:
import pandas as pd
from datetime import datetime
import os

In [2]:
# Load the CSV file into a pandas DataFrame
cPath = os.path.dirname(os.path.realpath('./CAE'))
file_path = cPath + '/Y2023.v2.csv'
Y3 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y3.set_index('Start', drop = False, inplace=True)
file_path = cPath + '/Y2024.csv'
Y4 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y4.set_index('Start', drop = False, inplace=True)
file_path = cPath + '/Y2025.csv'
Y5 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y5.set_index('Start', drop = False, inplace=True)
#file_path = '/home/homayoun/Dropbox/Projects/VisualCode/Python/CAE/Yearly.csv'
#dfy = pd.read_csv(file_path)
#print(df['Start'][0], df['USAGE'][0], df['End'][0], df['USAGE'][0])
#print(df['Start'][len(df)-1], df['USAGE'][len(df)-1], df['End'][len(df)-1], df['USAGE'][len(df)-1])
#print(type(df['End'][0]))
Y3['Delta'] = 0
Y4['Delta'] = 0
Y5['Delta'] = 0
# Adding a 'label' column to differentiate between OffPeak, OnPeak, and OnPeakHigh 
Y3['label'] = 'OffPeak'
Y4['label'] = 'OffPeak'
Y5['label'] = 'OffPeak'

In [3]:
# Find irregular data
def missing_data(data):
    e_td = pd.to_timedelta('15 minutes')
    for i in range(0, len(data)-1, 1):
        c_td = data['Start'][i+1] - data['Start'][i]
        if(e_td != c_td): 
           print (data['Start'][i], data['Start'][i+1], c_td)

missing_data(Y3)
missing_data(Y4)
missing_data(Y5)

/tmp/ipykernel_9039/4216920248.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  c_td = data['Start'][i+1] - data['Start'][i]
/tmp/ipykernel_9039/4216920248.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print (data['Start'][i], data['Start'][i+1], c_td)


2023-01-31 23:45:00 2023-02-01 04:00:00 0 days 04:15:00
2023-03-12 01:30:00 2023-03-12 02:45:00 0 days 01:15:00
2023-04-07 23:45:00 2023-04-08 04:00:00 0 days 04:15:00
2023-04-09 03:45:00 2023-04-10 00:00:00 0 days 20:15:00
2023-04-30 23:45:00 2023-05-01 04:00:00 0 days 04:15:00
2023-09-30 23:45:00 2023-10-01 04:00:00 0 days 04:15:00
2023-11-05 00:45:00 2023-11-05 00:45:00 0 days 00:00:00
2023-11-05 01:00:00 2023-11-05 01:00:00 0 days 00:00:00
2023-11-05 01:15:00 2023-11-05 01:15:00 0 days 00:00:00
2023-11-05 01:30:00 2023-11-05 01:30:00 0 days 00:00:00
2024-02-08 23:45:00 2024-02-10 00:00:00 1 days 00:15:00
2024-03-10 01:30:00 2024-03-10 02:45:00 0 days 01:15:00
2024-06-30 23:45:00 2024-07-01 04:00:00 0 days 04:15:00
2024-08-31 23:45:00 2024-09-26 05:00:00 25 days 05:15:00
2024-10-31 23:45:00 2024-11-01 05:00:00 0 days 05:15:00
2024-11-03 00:45:00 2024-11-03 00:45:00 0 days 00:00:00
2024-11-03 01:00:00 2024-11-03 01:00:00 0 days 00:00:00
2024-11-03 01:15:00 2024-11-03 01:15:00 0 days 

In [4]:
# Calculate the derivative of the usage
def calc_delta(data):
    e_td = pd.to_timedelta('15 minutes')
    delta = []
    for i in range(0, len(data)-1, 1):
        dUsage = data['USAGE'][i+1] - data['USAGE'][i]
        dt = data['Start'][i+1] - data['Start'][i]
        delta.append(dUsage / dt.total_seconds())
    delta.append(0)
    return delta 

Y3['Delta'] = calc_delta(Y3)
Y4['Delta'] = calc_delta(Y4)
Y5['Delta'] = calc_delta(Y5)

/tmp/ipykernel_9039/4111328373.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dUsage = data['USAGE'][i+1] - data['USAGE'][i]
/tmp/ipykernel_9039/4111328373.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dt = data['Start'][i+1] - data['Start'][i]
/tmp/ipykernel_9039/4111328373.py:8: RuntimeWarning: divide by zero encountered in divide
  delta.append(dUsage / dt.total_seconds())


In [ ]:
# This needs closer inspection, but it seems to be a way to set the labels based on the month and hour

def set_labels(data):
    data['label'] = 'w'  # Default label as winter
    month_mask = data.index.month.isin([5, 6, 7, 8, 9])  # May to September
    data.loc[month_mask, 'label'] = 's' # Summer months

    # Identify OnPeak based on winter or summer
    time_mask = data.index.hour.isin([18, 19, 20, 21])  # Summer OnPeak hours. Does not include 22
    label_mask = data['label'] == 's'
    data.loc[time_mask & label_mask, 'label'] = 'sOnPeak'  # Summer OnPeak
    time_mask = data.index.hour.isin([6, 7, 8, 9, 18, 19, 20, 21])  # OnPeak hours. Does not include 22
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify the OffPeak times
    label_mask = data['label'] == 's'
    data.loc[label_mask, 'label'] = 'OffPeak'  # Summer OffPeak 
    label_mask = data['label'] == 'w'
    data.loc[label_mask, 'label'] = 'OffPeak'  # Winter OffPeak
    label_mask = data.index.day_of_week.isin([5, 6])  # Saturday and Sunday
    data.loc[label_mask, 'label'] = 'OffPeak'  # Weekend OffPeak

    #Identify the high usage periods during OnPeak times
    data.loc[(data['label'] == 'wOnPeak') & (data['USAGE'] > 100), 'label'] = 'wOnPeakHigh'
    data.loc[(data['label'] == 'sOnPeak') & (data['USAGE'] > 100), 'label'] = 'sOnPeakHigh'
    return data

Y3 = set_labels(Y3)
print('Y3: ', Y3.value_counts('label'), '\n')
Y4 = set_labels(Y4)
print('Y4: ', Y4.value_counts('label'), '\n')
Y5 = set_labels(Y5)
print('Y5: ', Y5.value_counts('label'), '\n')

Y3:  label
OffPeak        28336
wOnPeakHigh     3422
wOnPeak         1378
sOnPeakHigh     1036
sOnPeak          724
Name: count, dtype: int64 

Y4:  label
OffPeak        26264
wOnPeakHigh     3733
wOnPeak         1131
sOnPeak          736
sOnPeakHigh      720
Name: count, dtype: int64 

Y5:  label
OffPeak        8684
wOnPeakHigh    1785
wOnPeak         935
Name: count, dtype: int64 



In [7]:
Y3.to_csv('Y3.labels.csv')
Y4.to_csv('Y4.labels.csv')
Y5.to_csv('Y5.labels.csv')